# Блокнот для воспроизведения результатов статьи

## Воспроизведение результатов с использованием предложенного бенчмарка и моделей

вывод используемых карточек (запускаю на компьюетере с 2мя А100, но 1на из них используется для подгрузки данных - особенность их бенчмарка, чтобы честнее замерить скорость&память)

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print(torch.__version__, 'compute capability:', torch.cuda.get_device_capability())

name, memory.total [MiB]
NVIDIA A100 80GB PCIe, 81920 MiB
NVIDIA A100 80GB PCIe, 81920 MiB
2.6.0+cu124 compute capability: (8, 0)


In [5]:
%ls
%cd ml-cross-entropy

hello.md  ml-cross-entropy/  out/  test.ipynb
/disk/10tb/home/pam-edelkin/test_cce/ml-cross-entropy


/disk/10tb/home/pam-edelkin/miniconda3/envs/cce_bench/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


### Запуск для Gemma2

In [6]:
!python -m benchmark --methods=cce,cce_kahan_full_c,cce_exact,torch_compile,baseline --test_data=gemma2 --n_iteration=50 --n_rep=1 --output=/disk/10tb/home/pam-edelkin/test_cce/out/gemma2.csv

config.json: 100%|████████████████████████████| 838/838 [00:00<00:00, 5.61MB/s]

tokenizer_config.json: 47.0kB [00:00, 66.8MB/s]

tokenizer.model:   0%|                             | 0.00/4.24M [00:00<?, ?B/s]
tokenizer.model: 100%|████████████████████| 4.24M/4.24M [00:01<00:00, 2.55MB/s]

tokenizer.json:   0%|                              | 0.00/17.5M [00:00<?, ?B/s]
tokenizer.json:  37%|███████▊             | 6.47M/17.5M [00:00<00:00, 12.7MB/s]
tokenizer.json: 100%|█████████████████████| 17.5M/17.5M [00:00<00:00, 18.9MB/s]

special_tokens_map.json: 100%|████████████████| 636/636 [00:00<00:00, 4.80MB/s]

model.safetensors.index.json: 24.2kB [00:00, 57.6MB/s]


model-00001-of-00002.safetensors:   0%|            | 0.00/4.99G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   3%|    | 134M/4.99G [00:01<00:54, 88.8MB/s]

model-00001-of-00002.safetensors:   5%|▎    | 268M/4.99G [00:02<00:32, 146MB/s]

model-00001-of-00002.safetensors:   8%|▍    | 402M/4.99G [00:02<00:24, 185MB/s]

model-

In [7]:
import pandas as pd
df = pd.read_csv('/disk/10tb/home/pam-edelkin/test_cce/out/gemma2.csv', index_col=0)
df['kind'] = df['kind'].map({'loss-fw': 'Loss', 'loss-bw': 'Gradient', 'loss-fw-bw': 'Loss+Gradient'})
t = df.pivot(index='method', columns=['kind'], values=['op_mem_mb', 'runtime_ms'])
t = t.swaplevel(axis=1).reindex(columns=pd.MultiIndex.from_product([['Loss', 'Gradient', 'Loss+Gradient'], ['op_mem_mb', 'runtime_ms']]))
t.round(1)

Loss             Gradient            Loss+Gradient  \
                 op_mem_mb runtime_ms op_mem_mb runtime_ms     op_mem_mb   
method                                                                     
baseline           24000.0       86.3   16000.0      129.1       28000.0   
cce                    1.0       80.4    1162.3      100.7        1163.3   
cce_exact              0.1       74.7    3447.3      419.4        3447.3   
cce_kahan_full_c       1.0       80.1    3448.3      307.8        3449.3   
torch_compile       4000.1       51.0    1160.9       88.8        5161.0   

                             
                 runtime_ms  
method                       
baseline              224.7  
cce                   185.4  
cce_exact             498.2  
cce_kahan_full_c      391.8  
torch_compile         145.0

### Запуск для llama3.2-3

In [12]:
!python -m benchmark --methods=cce,cce_kahan_full_c,cce_exact,torch_compile,baseline --test_data="llama3.2-3" --n_iteration=50 --n_rep=1 --output=/disk/10tb/home/pam-edelkin/test_cce/out/llama3_2_3.csv

Loading checkpoint shards: 100%|█████████████████| 2/2 [00:02<00:00,  1.16s/it]

generation_config.json: 100%|█████████████████| 189/189 [00:00<00:00, 1.56MB/s]

Benchmark kind:   0%|                                    | 0/3 [00:00<?, ?it/s]

Method:   0%|                                            | 0/5 [00:00<?, ?it/s]/disk/10tb/home/pam-edelkin/test_cce/ml-cross-entropy/benchmark/memory.py:59: CCEWarning: There is a known issue with CCE and Triton 3.2 (the version that ships with PyTorch 2.6) that can result in incorrect gradients. If possible, please verify that you are not impacted by this bug by trying a newer triton version (i.e. by installing PyTorch>2.6).
  t = f(*args, **kwds)
/disk/10tb/home/pam-edelkin/test_cce/ml-cross-entropy/benchmark/memory.py:92: CCEWarning: There is a known issue with CCE and Triton 3.2 (the version that ships with PyTorch 2.6) that can result in incorrect gradients. If possible, please verify that you are not impacted by this bug by trying a newer tr

In [13]:
import pandas as pd
df = pd.read_csv('/disk/10tb/home/pam-edelkin/test_cce/out/llama3_2_3.csv', index_col=0)
df['kind'] = df['kind'].map({'loss-fw': 'Loss', 'loss-bw': 'Gradient', 'loss-fw-bw': 'Loss+Gradient'})
t = df.pivot(index='method', columns=['kind'], values=['op_mem_mb', 'runtime_ms'])
t = t.swaplevel(axis=1).reindex(columns=pd.MultiIndex.from_product([['Loss', 'Gradient', 'Loss+Gradient'], ['op_mem_mb', 'runtime_ms']]))
t.round(1)

Loss             Gradient            Loss+Gradient  \
                 op_mem_mb runtime_ms op_mem_mb runtime_ms     op_mem_mb   
method                                                                     
baseline           10020.0       83.4    8016.0      130.6       12024.0   
cce                    0.6       60.9     800.2      149.2         800.7   
cce_exact              0.1       63.0    2351.7      535.1        2351.7   
cce_kahan_full_c       0.6       64.7    2352.2      447.9        2352.7   
torch_compile       2004.1       56.8     799.4      100.9        2803.5   

                             
                 runtime_ms  
method                       
baseline              218.1  
cce                   204.0  
cce_exact             592.7  
cce_kahan_full_c      500.8  
torch_compile         164.9

## Своя модель c их функиональносью для замеров

Идея затестировать интеграцию их ядер в самописную архитектуру модели (а не подгруженную с hf)

Но можель тут сразу адаптирована на подсчет лосса, чтобы можно было адекватно сравнить - веса одинаковые, поэтому пример немного игрушечный, но отличный для того, чтобы убедиться в толковости и производительности

### imports

In [1]:
import gc
import traceback
import warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
from cut_cross_entropy import linear_cross_entropy

In [2]:
torch.backends.cudnn.benchmark = True 
try:
    torch._functorch.config.donated_buffer = False
except (AttributeError, ImportError):
    pass

### model

In [6]:
class RNN(nn.Module):
    def __init__(self, vocab, embedding=256, hidden=512, layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab, embedding)
        self.rnn = nn.LSTM(embedding, hidden, layers, batch_first=True)
        self.head = nn.Linear(hidden, vocab, bias=False)
    
    def encode(self, x):
        h = self.embedding(x)
        h, _ = self.rnn(h)
        return h

    def forward_baseline(self, x, targets):
        h = self.encode(x)
        logits = self.head(h)
        return F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            targets.reshape(-1)
        )

    def forward_cce(self, x, targets):
        h = self.encode(x)
        return linear_cross_entropy(
            h.reshape(-1, h.size(-1)),
            self.head.weight,
            targets.reshape(-1)
        )
        
def time_measurements_frame(fn, n):
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    torch.cuda.synchronize()
    start.record()

    for i in range(n):
        fn()
    
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / n

def reset_peak():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

def peak_mb():
    return torch.cuda.max_memory_allocated() / 2**20

def bench(name, model, forward_fn, warmap_steps, time_steps):
    res = {"variant": name, "errors":{}}
    try:
        #warmup
        for i in range(warmap_steps):
            out = forward_fn()
            out.backward()
            del out
            model.zero_grad(set_to_none=True)
        torch.cuda.synchronize()
    
        # forward
        reset_peak()
        res["fwd_ms"] = time_measurements_frame(forward_fn, time_steps)
        res["fwd_mb"] = peak_mb()
    except Exception as e:
        res["errors"]["warmup/forward"] = f"{type(e).__name__}: {e}"
        traceback.print_exc()

    try:
        # backward
        out = forward_fn()
        torch.cuda.synchronize()
        reset_peak()
        res["bwd_ms"] = time_measurements_frame(
            lambda: out.backward(retain_graph=True), time_steps
        )
        res["bwd_mb"] = peak_mb()
        del out
        model.zero_grad(set_to_none=True)
    except Exception as e:
        res["errors"]["backward"] = f"{type(e).__name__}: {e}"
        traceback.print_exc()

    try:
        def forw_backw():
            out = forward_fn()
            out.backward(retain_graph=True)
            model.zero_grad(set_to_none=True)

        reset_peak()
        res["fwd_bwd_ms"] = time_measurements_frame(forw_backw, time_steps)
        res["fwd_bwd_mb"] = peak_mb()
    except Exception as e:
        res["errors"]["forward_and_backward"] = f"{type(e).__name__}: {e}"
        traceback.print_exc()

    return res

### Function for printing

In [7]:
COLS = [("fwd_ms", 2), ("bwd_ms", 2), ("fwd_bwd_ms", 2),
        ("fwd_mb", 1), ("bwd_mb", 1), ("fwd_bwd_mb", 1)]


def print_table(results):
    header = (f"{'variant':<20s} {'fwd ms':>9s} {'bwd ms':>9s} {'f+b ms':>9s} "
              f"{'fwd MiB':>9s} {'bwd MiB':>9s} {'f+b MiB':>9s}")
    print(header)
    print("-" * len(header))
    for r in results:
        cells = [f"{r[k]:>9.{d}f}" if k in r else f"{'—':>9s}" for k, d in COLS]
        print(f"{r['variant']:<20s} " + " ".join(cells))
    print("-" * len(header))

    for r in results:
        for phase, msg in r["errors"].items():
            print(f"[!] {r['variant']} / {phase}: {msg}")

    def get(v, k):
        for r in results:
            if r["variant"] == v and k in r:
                return r[k]
        return None

    for a, b in [("baseline+compile", "baseline"), ("cce", "baseline")]:
        ta, tb = get(a, "fwd_bwd_ms"), get(b, "fwd_bwd_ms")
        ma, mb = get(a, "fwd_bwd_mb"), get(b, "fwd_bwd_mb")
        if ta and tb and ma and mb:
            print(f"{a} vs {b}: время f+b {ta/tb:.2f}x, память f+b {mb/ma:.1f}x меньше")

### Запуск

In [9]:
torch.manual_seed(0)

B, T = 8, 512
V = 128000
n_warmup, n_iter = 5, 30

print(f"GPU: {torch.cuda.get_device_name(0)} | torch {torch.__version__}")
print(f"B={B}, T={T}, V={V} | N=B*T={B*T} | dtype=bf16")
print(f"логиты [{B*T} x {V}] bf16 = {B*T*V*2/2**20:.0f} MiB на тензор")
print()

model = RNN(V).cuda().to(torch.bfloat16)
model.train()

x = torch.randint(0, V, (B, T), device="cuda")
y = torch.randint(0, V, (B, T), device="cuda")

compiled_baseline = torch.compile(model.forward_baseline, dynamic=False)

variants = [
    ("baseline",          lambda: model.forward_baseline(x, y)),
    ("baseline+compile",  lambda: compiled_baseline(x, y)),
    ("cce",               lambda: model.forward_cce(x, y)),
]

results = [bench(n, model, f, n_warmup, n_iter) for n, f in variants]
print()
print_table(results)

GPU: NVIDIA A100 80GB PCIe | torch 2.6.0+cu124
B=8, T=512, V=128000 | N=B*T=4096 | dtype=bf16
логиты [4096 x 128000] bf16 = 1000 MiB на тензор



/tmp/ipykernel_3287172/1794518414.py:23: CCEWarning: There is a known issue with CCE and Triton 3.2 (the version that ships with PyTorch 2.6) that can result in incorrect gradients. If possible, please verify that you are not impacted by this bug by trying a newer triton version (i.e. by installing PyTorch>2.6).
  return linear_cross_entropy(
/tmp/ipykernel_3287172/1794518414.py:23: CCEWarning: There is a known issue with CCE and Triton 3.2 (the version that ships with PyTorch 2.6) that can result in incorrect gradients. If possible, please verify that you are not impacted by this bug by trying a newer triton version (i.e. by installing PyTorch>2.6).
  return linear_cross_entropy(



variant                 fwd ms    bwd ms    f+b ms   fwd MiB   bwd MiB   f+b MiB
--------------------------------------------------------------------------------
baseline                 65.14    181.31    254.84    2337.8    3528.4    3333.8
baseline+compile         65.85    182.39    253.00    1337.9    2658.4    2463.9
cce                      66.34    183.64    252.63     338.4     660.4     538.5
--------------------------------------------------------------------------------
baseline+compile vs baseline: время f+b 0.99x, память f+b 1.4x меньше
cce vs baseline: время f+b 0.99x, память f+b 6.2x меньше


---